# Chapter 17 — Tools in Depth

**Companion to *Pi Agents*** · [`Pi Agents` chapter 17](https://programmer.ie/books/pi/17-chapter/)

| | |
|---|---|
| Chapter | `17-chapter.md` · weight 17 · `/books/pi/17-chapter/` |
| Notebook | `notebooks/pi/17-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Do nested calls pass the same gate, and does `deferred` exposure keep a tool
registered but inactive?

The chapter's point: `deferred` exposure means "registered but not declared to the
model" — it stays callable from another tool's execute. And every tool call,
whether from the model or from `ctx.executeTool()`, passes through the same
`tool_call` event.


## What this notebook establishes

- A `deferred`-exposure tool is **registered but not active** at registration — it does not appear in `getActiveToolNames()` until activated.
- Nested calls via `ctx.executeTool()` **pass through the same `tool_call` gate** as model-issued calls — the spy sees all three in order.
- A tool with `readOnlyHint: true` and `openWorldHint: false` passes the approval gate; one with **no hints is treated as destructive and open-world** and blocked when there is no UI.
- A non-zero exit from a command is **data, not a failed tool result** — the tool returns `isError: false` with the exit code in details.
- StructuredContent is returned alongside the model-facing content.

## What this notebook does **not** establish

- **Annotations are hints, not permission** — a server that lies about `readOnlyHint` passes the gate (`ch17-lim3` / `ch21-lim2`).
- **`registerToolRenderer` arrived in 1.0.1** and was not exercised in a running session by the chapter's suite.
- **No measurement** of how often a model calls a tool it was offered.
- **Parallel tool calls** from one assistant message can race — handlers must be independent.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — real sessions with the chapter's tools, approval, and dynamic-activation extensions, driven by a scripted model.
* **Evidence scope:** `in_process_runtime`. The gate, exposure, activation and approval rules are Pi's.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(17))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/17-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: a command tool returns data, not errors

`run_checks` runs `pnpm typecheck` (and optionally tests). A non-zero exit is a
result, not a tool failure.

In [3]:
TOOLS = pinb.driver_source("ch17-tools.ts")
tools = pinb.driver(TOOLS, label="ch17-tools", timeout=300)
print(pinb.md_table(
    ["tool", "isError", "exit code in details?"],
    [["run_checks (skipTests)", tools["basic"]["isError"], "yes" if (tools["basic"].get("details", {}) or {}).get("results", [{}])[0].get("exit_code") is not None else "no"]],
))

| tool | isError | exit code in details? |
|---|---|---|
| run_checks (skipTests) | False | yes |


In [4]:
pinb.show_checks([
    pinb.check("run_checks returns isError: false even if the command exits non-zero",
               not tools["basic"]["isError"], "isError: false"),
    pinb.check("the exit code is in details.results[0].exit_code",
               (tools["basic"].get("details", {}) or {}).get("results", [{}])[0].get("exit_code") is not None, "present"),
])

== Assertions ==
  assertion  observed
----  --------------------------------------------------------------------  --------------
PASS  run_checks returns isError: false even if the command exits non-zero  isError: false
PASS  the exit code is in details.results[0].exit_code  present

2/2 assertions held.


## StructuredContent: the tool returns it, but the toolResult message carries content and details

The chapter's test is named "structuredContent is returned" but only asserts on the
model-facing `content` text. The `structuredContent` field from the tool's return
value is not automatically copied to the toolResult message.

In [5]:
print(pinb.md_table(
    ["tool", "content text"],
    [["failing_tests", tools["structured"]["content"][:60]]],
))

| tool | content text |
|---|---|
| failing_tests | 1 file(s) with failing tests. |


In [6]:
pinb.show_checks([
    pinb.check("failing_tests returns content about failing tests",
               "failing tests" in tools["structured"]["content"], "content present"),
])

== Assertions ==
  assertion  observed
----  -------------------------------------------------  ---------------
PASS  failing_tests returns content about failing tests  content present

1/1 assertions held.


## Nested calls: review_diff runs two other tools and reads their outcomes

`review_diff` calls `run_checks` and `failing_tests` via `ctx.executeTool()`.
Both nested calls pass through the same gate.

In [7]:
print(pinb.md_table(
    ["tool", "isError", "nested tools seen"],
    [["review_diff", tools["nested"]["isError"], ", ".join(tools["nested"]["nested"])]],
))

| tool | isError | nested tools seen |
|---|---|---|
| review_diff | False | run_checks, failing_tests |


In [8]:
print("Gate order (what the spy saw):", " -> ".join(tools["nestedGate"]["seen"]))
print()
pinb.show_checks([
    pinb.check("nested calls pass through the same tool_call gate in order",
               tools["nestedGate"]["seen"] == ["review_diff", "run_checks", "failing_tests"],
               " -> ".join(tools["nestedGate"]["seen"])),
    pinb.check("review_diff reads structuredContent from failing_tests",
               "src/auth/refresh.ts" in tools["nested"]["content"][1]["text"],
               "structuredContent came through"),
])

Gate order (what the spy saw): review_diff -> run_checks -> failing_tests

== Assertions ==
  assertion  observed
----  ----------------------------------------------------------  ------------------------------------------
PASS  nested calls pass through the same tool_call gate in order  review_diff -> run_checks -> failing_tests
PASS  review_diff reads structuredContent from failing_tests  structuredContent came through

2/2 assertions held.


## Approval gate: annotations are hints, not permission

`run_checks` declares `readOnlyHint: true, openWorldHint: false` — no approval
needed. `failing_tests` declares **no hints** — treated as destructive and
open-world, blocked without a UI.

In [9]:
print(pinb.md_table(
    ["tool", "isError", "why"],
    [["run_checks (readOnlyHint)", "no" if not tools["approval"]["checksIsError"] else "yes", "no approval needed"],
     ["failing_tests (no hints)", "yes" if tools["approval"]["failingIsError"] else "no", "blocked without UI"]],
))

| tool | isError | why |
|---|---|---|
| run_checks (readOnlyHint) | no | no approval needed |
| failing_tests (no hints) | yes | blocked without UI |


In [10]:
pinb.show_checks([
    pinb.check("run_checks with readOnlyHint passes approval",
               not tools["approval"]["checksIsError"], "allowed"),
    pinb.check("failing_tests with no hints is blocked without UI",
               tools["approval"]["failingIsError"], "blocked"),
    pinb.check("the blocked result has a reason the model can read",
               "not approved" in tools["approval"]["failingText"], "reason present"),
])

== Assertions ==
  assertion  observed
----  --------------------------------------------------  --------------
PASS  run_checks with readOnlyHint passes approval  allowed
PASS  failing_tests with no hints is blocked without UI  blocked
PASS  the blocked result has a reason the model can read  reason present

3/3 assertions held.


## Dynamic activation: deferred exposure keeps a tool registered but inactive

`failing_tests` is registered with `exposure: "deferred"`. It is not active
until `review_diff` is called, which activates it via `setActiveTools()`.

In [11]:
print(pinb.md_table(
    ["moment", "failing_tests in getActiveToolNames?"],
    [["at registration", "yes" if "failing_tests" in tools["dynamic"]["initiallyActive"] else "no"],
     ["after review_diff called", "yes" if "failing_tests" in tools["dynamic"]["afterActive"] else "no"]],
))

| moment | failing_tests in getActiveToolNames? |
|---|---|
| at registration | no |
| after review_diff called | yes |


In [12]:
pinb.show_checks([
    pinb.check("deferred tool is NOT active at registration",
               "failing_tests" not in tools["dynamic"]["initiallyActive"], "inactive initially"),
    pinb.check("deferred tool IS registered at registration",
               "failing_tests" in tools["dynamic"]["initiallyRegistered"], "registered but not active"),
    pinb.check("after review_diff, the deferred tool is activated",
               "failing_tests" in tools["dynamic"]["afterActive"], "active now"),
    pinb.check("and it is callable from another tool",
               not tools["dynamic"]["reviewDiffIsError"], "no error"),
])

== Assertions ==
  assertion  observed
----  -------------------------------------------------  -------------------------
PASS  deferred tool is NOT active at registration  inactive initially
PASS  deferred tool IS registered at registration  registered but not active
PASS  after review_diff, the deferred tool is activated  active now
PASS  and it is callable from another tool  no error

4/4 assertions held.


## The chapter's own tests

All seven tests from the chapter's test file.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [13]:
PATTERNS = 'ch17-tools/tools.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch17-tools/tools.test.ts', show="nested")

$ node --test ch17-tools/tools.test.ts
  PASS  7 passed, 0 failed, 0 skipped  in 1064 ms

  tests matching 'nested':
    ok   nested calls: review_diff runs two other tools and reads their outcomes
    ok   nested calls pass through the same gate as model-issued calls


## Your turn: predict, then run

**Predict first.** Change `failing_tests` to `exposure: "hidden"` instead of
`"deferred"`. Can `review_diff` still call it via `ctx.executeTool()`?

**Then change one input.** The approval extension checks `destructiveHint`,
`readOnlyHint`, `openWorldHint`. What happens if a tool declares
`destructiveHint: false` but no `readOnlyHint`?

**Predict the boundary.** Parallel tool calls from one assistant message can run
in any order. The spy in `nestedGate` saw them sequentially because the faux
provider serialises them — what happens with a real provider that streams two
tool calls at once?

In [14]:
print("Predictions:")
print("  1. hidden: no — a hidden tool is unreachable, even from executeTool().")
print("  2. destructiveHint: false alone is not enough — readOnlyHint is what the gate checks.")
print("  3. Parallel calls: handlers must be independent; order is not guaranteed.")
print()
print("Observed above:")
print(f"  deferred initially active: {"failing_tests" in tools["dynamic"]["initiallyActive"]}")
print(f"  deferred after review_diff: {"failing_tests" in tools["dynamic"]["afterActive"]}")
print(f"  nested gate order: {" -> ".join(tools["nestedGate"]["seen"])}")
print()
assert "failing_tests" not in tools["dynamic"]["initiallyActive"]
assert "failing_tests" in tools["dynamic"]["afterActive"]
assert tools["nestedGate"]["seen"] == ["review_diff", "run_checks", "failing_tests"]
print("held: deferred means registered-but-inactive, and nested calls pass the same gate")

Predictions:
  1. hidden: no — a hidden tool is unreachable, even from executeTool().
  2. destructiveHint: false alone is not enough — readOnlyHint is what the gate checks.
  3. Parallel calls: handlers must be independent; order is not guaranteed.

Observed above:
  deferred initially active: False
  deferred after review_diff: True
  nested gate order: review_diff -> run_checks -> failing_tests

held: deferred means registered-but-inactive, and nested calls pass the same gate


## Interpretation

A tool is three things, and the chapter shows the boundaries:

| Property | What it controls | The boundary |
|---|---|---|
| `exposure: "direct"` | Active at registration, declared to model | The default for built-ins |
| `exposure: "deferred"` | Registered, not active, not declared | Activated by `setActiveTools()`; still callable from `executeTool()` |
| `exposure: "codemode"` | Registered, not declared, callable from scripts | The MCP default (ch 21) |
| `exposure: "hidden"` | Registered, unreachable | Never callable |

| Annotation | What it means for approval |
|---|---|
| `readOnlyHint: true, openWorldHint: false` | No approval needed — read-only, closed world |
| No hints declared | **Treated as destructive and open-world** — blocked without UI |
| Lying `readOnlyHint` | **Passes the gate** — annotations are unverified by Pi |

The practical rules:
* **Every tool call passes the same gate** — model calls and `ctx.executeTool()` both fire `tool_call`.
* **Deferred is the pattern for "internal" tools** — the model doesn't see them, but your tools can call them.
* **Annotations are hints, not permission** — chapter 21 shows a lying MCP server passes the gate.
* **Catch inside handlers** — a throwing `tool_call` handler blocks the tool (fail-safe from chapter 16).

## Sources, execution mode and limitations

**Execution mode:** **Run** — real sessions with the chapter's tools, approval, and dynamic-activation extensions, driven by a scripted model.

**Evidence scope:** `in_process_runtime`. The gate, exposure, activation and approval rules are Pi's.

### What was read or run

- **Ran** `drivers/ch17-tools.ts`: six real sessions covering basic, structuredContent, nested, nested-gate, approval, and dynamic activation.
- **Ran** `ch17-tools/tools.test.ts` (7 tests).
- **Read** `examples/evidence.json`, chapter 17 rows (7 claims).

### Limitations

- **Annotations are hints, not permission.** A tool or server that lies about `readOnlyHint` passes the gate (`ch17-lim3` / `ch21-lim2`).
- **`registerToolRenderer` is documented-not-run** by the chapter's suite (`ch17-lim3`); it arrived in 1.0.1.
- **No measurement** of how often a model calls a tool it was offered.
- **Parallel tool calls** from one assistant message can race — the faux provider serialises them; a real provider may not.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
